# T07 — 균형·자세 제어와 협응

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/I-gait-theory/note-T07.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/I-gait-theory/note-T07.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 본문

### 7. 협응을 재는 방법과 그 가정

아래 코드는 두 규칙을 작은 가상 신호로 확인한다. 참 위상차가 90°로 일정한 두 신호에서, 순수 사인파면 CRP가 90°로 일정하지만 2차 고조파를 섞으면 크게 출렁인다.

In [ ]:
import numpy as np

fs = 100                                   # 표본/초, 한 주기 = 1 s로 둔 가상 신호
t = np.arange(0, 3, 1 / fs)                # 3주기를 만들고 가운데 주기만 본다
w = 2 * np.pi
mid = (t >= 1) & (t < 2)

def phase_angle(x):
    """[RM] 13장 방식: 각도는 min-max로 -1~1, 각속도는 |최대|로 나눈 뒤 atan2(속도, 각도)."""
    v = np.gradient(x, t)
    xn = 2 * (x - x.min()) / (x.max() - x.min()) - 1
    vn = v / np.abs(v).max()
    return np.degrees(np.arctan2(vn, xn))

def crp(a, b):
    d = np.abs(phase_angle(a) - phase_angle(b)) % 360
    return np.where(d > 180, 360 - d, d)[mid]   # 차이만 0-180°로 접는다

lag = np.pi / 2                                 # 두 신호의 참 위상차 90°
a1, b1 = np.sin(w * t), np.sin(w * t - lag)     # 순수 사인파 한 쌍
h = lambda u: np.sin(u) + 0.5 * np.sin(2 * u)   # 2차 고조파가 섞인 비정현 파형
a2, b2 = h(w * t), h(w * t - lag)

for name, (a, b) in {"사인파": (a1, b1), "비정현파": (a2, b2)}.items():
    c = crp(a, b)
    print(f"{name}: CRP 최소 {c.min():5.1f}°, 최대 {c.max():5.1f}°, 평균 {c.mean():5.1f}°")

ang = np.radians([359.0, 1.0])                  # 원형 자료의 평균
print("산술평균:", np.degrees(ang).mean(),
      "/ 원형평균:", round(float(np.degrees(np.arctan2(np.sin(ang).mean(), np.cos(ang).mean()))), 3) + 0.0,
      "/ 평균 벡터 길이 r:", round(np.hypot(np.sin(ang).mean(), np.cos(ang).mean()), 4))